# Ocean Visualization - Variable Tests

Test suite for variables, metadata, and derived variable calculations.

In [1]:
import sys
sys.path.insert(0, '..')

from ocean_viz import variables
import xarray as xr
import numpy as np

ModuleNotFoundError: No module named 'imageio'

## Part 1: Variable Catalog

In [ ]:
# List available variables
available = variables.get_available_variables()
print(f"Available variables ({len(available)}):")
for var in sorted(available):
    print(f"  - {var}")

In [ ]:
# Get metadata for SST
sst_meta = variables.get_variable_metadata('sst')
print("SST Metadata:")
for key, value in sst_meta.items():
    print(f"  {key}: {value}")

In [ ]:
# Check variable properties
print("\nVariable Properties:")
print(f"  SST is 3D: {variables.is_3d_variable('sst')}")
print(f"  SST is derived: {variables.is_derived_variable('sst')}")
print(f"  Current speed is 3D: {variables.is_3d_variable('current_speed')}")
print(f"  Current speed is derived: {variables.is_derived_variable('current_speed')}")

In [ ]:
# Get default color scales
print("\nDefault Color Scales:")
for var in ['sst', 'u_current', 'salinity', 'current_speed']:
    scale = variables.get_default_color_scale(var)
    print(f"  {var}: vmin={scale['vmin']}, vmax={scale['vmax']}, cmap={scale['cmap']}")

## Part 2: Derived Variables Test

In [ ]:
from ocean_viz.derived import (
    calculate_current_speed,
    calculate_sst_gradient,
    calculate_sst_anomaly,
    calculate_thermal_habitat_mask,
)

# Create synthetic test dataset
np.random.seed(42)
lon = np.linspace(160, 235, 50)
lat = np.linspace(40, 68, 40)
time = np.arange(5)

# Create synthetic data
u_data = np.random.rand(5, 40, 50) * 0.5 - 0.25  # m/s
v_data = np.random.rand(5, 40, 50) * 0.5 - 0.25  # m/s
sst_data = np.random.rand(5, 40, 50) * 10 + 5  # 5-15°C

ds = xr.Dataset(
    {
        'uo': (['time', 'lat', 'lon'], u_data),
        'vo': (['time', 'lat', 'lon'], v_data),
        'thetao': (['time', 'lat', 'lon'], sst_data),
    },
    coords={
        'time': time,
        'lat': lat,
        'lon': lon,
    }
)

print("Synthetic dataset created:")
print(ds)

In [ ]:
# Calculate current speed
speed = calculate_current_speed(ds)
print(f"\nCurrent speed:")
print(f"  Shape: {speed.shape}")
print(f"  Min: {float(speed.min()):.3f} m/s")
print(f"  Max: {float(speed.max()):.3f} m/s")
print(f"  Mean: {float(speed.mean()):.3f} m/s")

In [ ]:
# Calculate SST anomaly
anomaly = calculate_sst_anomaly(ds)
print(f"\nSST anomaly (relative to temporal mean):")
print(f"  Shape: {anomaly.shape}")
print(f"  Min: {float(anomaly.min()):.3f}°C")
print(f"  Max: {float(anomaly.max()):.3f}°C")
print(f"  Mean: {float(anomaly.mean()):.6f}°C (should be ~0)")

In [ ]:
# Calculate thermal habitat mask
mask = calculate_thermal_habitat_mask(ds, temp_min=5, temp_max=15)
print(f"\nThermal habitat mask (5-15°C):")
print(f"  Shape: {mask.shape}")
print(f"  Suitable cells: {int(mask.sum())} / {mask.size}")
print(f"  Percentage: {100 * float(mask.mean()):.1f}%")

## Part 3: Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Current speed
speed_t0 = speed.isel(time=0)
speed_t0.plot(ax=axes[0, 0], cmap='plasma')
axes[0, 0].set_title('Current Speed (t=0)')

# Plot 2: SST
sst_t0 = ds['thetao'].isel(time=0)
sst_t0.plot(ax=axes[0, 1], cmap='turbo', vmin=5, vmax=15)
axes[0, 1].set_title('Sea Surface Temperature (t=0)')

# Plot 3: SST anomaly
anomaly_t0 = anomaly.isel(time=0)
anomaly_t0.plot(ax=axes[1, 0], cmap='RdBu_r')
axes[1, 0].set_title('SST Anomaly (t=0)')

# Plot 4: Thermal habitat mask
mask_t0 = mask.isel(time=0)
mask_t0.plot(ax=axes[1, 1], cmap='RdYlGn', vmin=0, vmax=1)
axes[1, 1].set_title('Thermal Habitat Mask (5-15°C, t=0)')

plt.tight_layout()
plt.show()